# Phase 12: Late Delivery Risk Decision Tree Classifier
## Marketplace Profitability & Delivery Intelligence Engine

This notebook evaluates predictive modeling for order delivery delays under strict data governance:
- **Operational Prediction Moment**: Carrier dispatch scan (`order_delivered_carrier_date`).
- **Strict Anti-Leakage Boundary**: Zero post-dispatch features (delivery dates, lead times, review scores).
- **Temporal Validation**: Chronological 80/20 train/test split reflecting real production deployment.
- **Class Imbalance Strategy**: Balanced class weighting on ~8.1% late deliveries.
- **Interpretability**: White-box decision tree with feature importances and extracted rules.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is accessible
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import joblib

from src.marketplace_intelligence.modeling import LateDeliveryPredictor, ModelFeatures

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11

predictor = LateDeliveryPredictor(processed_dir='../data/processed', docs_dir='../docs', models_dir='../models')
df_train, df_test, X_train, y_train, X_test, y_test = predictor.load_and_split_data(train_ratio=0.80)
print(f'Train size: {len(X_train):,} orders | Test size: {len(X_test):,} orders')

### 1. Temporal Split & Class Distribution Verification
In logistics forecasting, random cross-validation leaks future route optimizations and seasonal patterns into the past. We strictly split chronologically.

In [ ]:
print('Train Date Range:', df_train['order_purchase_timestamp'].min(), 'to', df_train['order_purchase_timestamp'].max())
print('Test Date Range: ', df_test['order_purchase_timestamp'].min(), 'to', df_test['order_purchase_timestamp'].max())
print(f'Train Late Rate:   {y_train.mean():.2%}')
print(f'Test Late Rate:    {y_test.mean():.2%}')
assert df_train['order_purchase_timestamp'].max() <= df_test['order_purchase_timestamp'].min(), 'Temporal leakage detected!'
print('Temporal anti-leakage assertion PASSED!')

### 2. Model Training & Benchmark Comparison
We compare three distinct modeling approaches on the chronological test set:
1. **Dummy Baseline** (Stratified random guessing matching training distribution)
2. **Unweighted Decision Tree** (`max_depth=6`, standard Gini split)
3. **Balanced Decision Tree** (`class_weight="balanced"`, penalized false negatives)

In [ ]:
pipeline_balanced = predictor.build_pipeline(max_depth=6, min_samples_leaf=100, class_weight='balanced')
metrics = predictor.evaluate_models(X_train, y_train, X_test, y_test, pipeline_balanced)
metrics_df = pd.DataFrame(metrics).T
display(metrics_df)

### 3. Feature Importance & Domain Attribution
Decision trees provide exact Gini impurity reduction percentages for every feature split.

In [ ]:
importances = predictor.extract_feature_importances(pipeline_balanced)
print('Top 10 Feature Importances:')
print((importances.head(10) * 100.0).round(2).astype(str) + '%')

### 4. Decision Threshold & Financial Cost Optimization
In operations, missing a late delivery costs ~$20 BRL (voucher + customer service overhead), whereas an automated SMS alert costs ~$1 BRL. We analyze cost curves across thresholds.

In [ ]:
threshold_df = predictor.compute_threshold_table(pipeline_balanced, X_test, y_test)
display(threshold_df)

### 5. Plain-Text Decision Rules
White-box business logic extracted from the top 3 layers of the trained decision tree:

In [ ]:
rules = predictor.extract_tree_rules(pipeline_balanced, max_depth=3)
print(rules)